# Notebook 3: Taking it home

**Framework:** Unsloth · **Model:** `unsloth/Qwen3.5-0.8B` · **Time:** ~25 minutes · **GPU:** free Colab T4

This notebook answers two questions: how do I **keep** what I've made, and how do I do this with **my own data**? It's also meant as a reference to come back to after the workshop.

**What you'll do**
1. Reload your pirate adapter from Google Drive, and export the model to a file you can run on your own laptop.
2. Run a template that fine-tunes the model on a CSV file. In the workshop it uses a sample dataset; at home you point it at your own.
3. Go through a checklist of common pitfalls.
4. See where to go next: bigger computers, bigger models, and how other researchers have used fine-tuning.

**Learning outcomes covered:** 2 (preparing data), 3 (LoRA), 4 (evaluation).

> **DRAFT NOTE (facilitator timing, remove before publishing):**
> 0 Setup (install during the break) · 1 Reload and export ~5 min (+ Ollama screen share) · 2 Template ~7 min (run 2a–2d live, talk through 2e–2i) · 3 Pitfalls ~5 min · 4 Where next ~5 min · buffer ~3 min. If earlier notebooks overran, talk through 3 and 4 quickly and leave them for self-study.

### Before you start: turn on the GPU
In Colab, go to **Runtime → Change runtime type**, choose **T4 GPU**, and click **Save**.

## 0. Setup

The same install as Notebook 1. **Run it now**: it takes a few minutes.

> **DRAFT NOTE:** Keep these pins identical to Notebook 1's install cell once that is pinned after the T4 test.

In [ ]:
%%capture
!pip install "unsloth==2026.11.3" "unsloth_zoo==2026.11.3"
!unsloth install-kernels

Connect Google Drive, so this notebook can read what you saved in Notebooks 1 and 2 and keep what you make here. Colab asks you to allow access.

Everything from the workshop lives in one folder in your Drive: `fine-tuning-workshop`.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/fine-tuning-workshop"
!mkdir -p {DRIVE_DIR}
!ls {DRIVE_DIR}

### Helpers from Notebook 1

*Hidden code: run it.* These are the `chat` and `ask_many` functions from Notebook 1, unchanged.

In [ ]:
# @title Helpers from Notebook 1: `chat` and `ask_many` (double-click to see the code)
import torch
from unsloth import FastLanguageModel

SEED = 3407
MODEL_NAME = "unsloth/Qwen3.5-0.8B"


def chat(question, system_prompt=None, max_new_tokens=200):
    # 1. Build the conversation
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})

    # 2. Apply the chat template, then turn the text into tokens on the GPU
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text=prompt, return_tensors="pt").to("cuda")

    # 3. Generate a reply
    torch.manual_seed(SEED)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                            temperature=0.7, do_sample=True)

    # 4. Keep only the new tokens (the reply), and turn them back into text
    reply_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply_tokens, skip_special_tokens=True).strip()


def ask_many(model, tokenizer, questions, system_prompt=None, max_new_tokens=200, batch_size=16):
    replies = []
    for start in range(0, len(questions), batch_size):
        batch = questions[start:start + batch_size]

        prompts = []
        for q in batch:
            messages = [{"role": "system", "content": system_prompt}] if system_prompt else []
            messages.append({"role": "user", "content": q})
            prompts.append(tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
                enable_thinking=False))   # some Qwen models "think" out loud first unless told not to

        # Questions have different lengths, so the shorter ones are padded with filler.
        # Padding goes on the left, so every prompt ends exactly where its reply begins.
        inputs = tokenizer(text=prompts, return_tensors="pt",
                           padding=True, padding_side="left").to("cuda")

        torch.manual_seed(SEED)
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                temperature=0.7, do_sample=True)
        reply_tokens = output[:, inputs["input_ids"].shape[1]:]
        replies += tokenizer.batch_decode(reply_tokens, skip_special_tokens=True)

    return [r.strip() for r in replies]

## 1. Reload and export your pirate model

Colab deletes everything when the session ends, which is why section 7 of Notebook 1 saved your adapter to Drive. Here you load it back, check it still talks like a pirate, and export it to run on your own computer.

### 1a. Load the base model and attach your adapter

Your adapter only stores the small LoRA matrices, plus the name of the base model it belongs to. Unsloth reads that name, downloads the base model, and attaches the adapter.

If your adapter isn't in Drive (for example, your training didn't finish), the cell downloads a copy we trained before the workshop.

In [ ]:
import gdown

PIRATE_LORA_DIR = f"{DRIVE_DIR}/pirate_lora"
FALLBACK_URL = "https://drive.google.com/drive/folders/FALLBACK_FOLDER_ID"   # TODO: workshop's shared folder

if os.path.exists(f"{PIRATE_LORA_DIR}/adapter_config.json"):
    adapter_dir = PIRATE_LORA_DIR
    print("Using your adapter from Drive")
else:
    adapter_dir = "/content/pirate_lora_fallback"
    gdown.download_folder(FALLBACK_URL, output=adapter_dir, quiet=True)
    print("No adapter in your Drive, so using the workshop's copy")

model, tokenizer = FastLanguageModel.from_pretrained(
    adapter_dir,
    max_seq_length=1024,
    load_in_4bit=False,
)
FastLanguageModel.for_inference(model)

> **DRAFT NOTE:** (a) Confirm on the T4 that `FastLanguageModel.from_pretrained(<adapter folder>)` loads Qwen3.5 plus the adapter (same `FastLanguageModel` vs `FastVisionModel` question as Notebook 1). (b) Create the shared Drive folder with the presenter's adapter, set it to "Anyone with the link can view", and put its URL in `FALLBACK_URL`. Use the same folder for Notebook 1's fallback cell. (c) Notebook 1 section 7 must save to `{DRIVE_DIR}/pirate_lora`.

Ask it a couple of questions with no system prompt. The answers should be in pirate speak, as at the end of Notebook 1.

In [ ]:
for q in ["What is photosynthesis?", "Write a two-line abstract for a paper on soil bacteria."]:
    print(f"Q: {q}\nA: {chat(q)}\n" + "-" * 60)

You could also push the adapter to the Hugging Face Hub to share it. We don't do that here: anything on a public hub is public, so check your data and ethics approvals first.

### 1b. Export to GGUF

**GGUF** is a single-file model format used by tools that run models on ordinary laptops, such as **Ollama** and **llama.cpp**. Exporting merges the adapter into the base model and saves the result as one file.

We also **quantise** it: store the weights with fewer bits (about 4 instead of 16). `q4_k_m` makes the file about 0.5 GB instead of 1.6 GB, with a small loss in quality. It's the usual format for Ollama. Use `q8_0` (about 0.9 GB) if you want quality closer to the original.

The first export takes several minutes, because Unsloth downloads and builds llama.cpp, which does the conversion.

> **DRAFT NOTE:** Time this on a T4 (llama.cpp build plus conversion), and check that llama.cpp supports Qwen3.5's hybrid (linear-attention) layers. If it's too slow or fails, show this cell without running it and give learners a link to the presenter's GGUF in the shared Drive folder instead. Check where Unsloth 2026.11.3 writes the `.gguf` file; the `glob` below allows for both a subfolder and a sibling file.

In [ ]:
model.save_pretrained_gguf("pirate_gguf", tokenizer, quantization_method="q4_k_m")

Copy the GGUF file to Drive under a clear name, and write a **Modelfile** next to it. A Modelfile tells Ollama which model file to use and which settings to run it with.

In [ ]:
import glob
import shutil

GGUF_DIR = f"{DRIVE_DIR}/pirate_gguf"
GGUF_NAME = "pirate-qwen3.5-0.8b-Q4_K_M.gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

found = [f for f in glob.glob("pirate_gguf*/**/*.gguf", recursive=True) + glob.glob("pirate_gguf*.gguf")
         if "q4_k_m" in f.lower()]
print("Exported:", found)
shutil.copy(found[0], f"{GGUF_DIR}/{GGUF_NAME}")

with open(f"{GGUF_DIR}/Modelfile", "w") as f:
    f.write(f"FROM ./{GGUF_NAME}\n")
    f.write("PARAMETER temperature 0.7\n")

!ls -lh {GGUF_DIR}

### 1c. Run it on your own computer with Ollama

*The presenter now shows this on their laptop with the same file.*

To do it yourself afterwards:

1. Install **Ollama** from [ollama.com](https://ollama.com).
2. In Google Drive, download the `fine-tuning-workshop/pirate_gguf` folder (the `.gguf` file and the `Modelfile`), and unzip it if Drive zips it.
3. Open a terminal in that folder and run:
   ```bash
   ollama create pirate -f Modelfile
   ollama run pirate
   ```
4. Type a question. Type `/bye` to quit.

The model now runs entirely on your computer, with no internet connection needed and no data leaving your machine.

**llama.cpp** is the lower-level option that Ollama is built on: `llama-cli -m pirate-qwen3.5-0.8b-Q4_K_M.gguf -cnv`.

> **DRAFT NOTE:** Rehearse with a GGUF exported from Colab. Check that Ollama supports Qwen3.5 and picks up the chat template from the GGUF's metadata; if replies look garbled, add a `TEMPLATE` block to the Modelfile (or use the Modelfile Unsloth writes next to the GGUF, if it does). Prepare the demo GGUF before the session: don't download a learner's file live.

Free the GPU before section 2.

In [ ]:
import gc

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 2. Template: fine-tune on your own data

This is the Notebook 1 workflow, driven by a few settings instead of written for one dataset. You give it a **CSV file** with an input column and a target column; it checks the data, formats it, trains a LoRA adapter, and compares the model before and after.

**Which path should you use?**
- **Text in, text out** (labels, extracted fields, rewritten text): this template, with Unsloth.
- **Sequences or numeric labels** (DNA, proteins, scores): follow Notebook 2 (Transformers + PEFT).

**The sample dataset** is 400 arXiv abstracts, each labelled with its field: Astrophysics, Quantitative biology, Economics or Mathematics (100 of each). It shows that a chat model can learn to reply with a label. arXiv's abstracts are released under CC0 (public domain). See [`data/arxiv_abstracts_README.md`](https://github.com/UoA-eResearch/fine-tuning_workshop/blob/main/data/arxiv_abstracts_README.md) for how it was made.

**In the workshop:** run 2a to 2d. Sections 2e to 2i train and evaluate the model, so run them at home on your own data. They take about 10 minutes on a T4.

### 2a. Edit me

These four settings are all you need to change for your own data.
- `CSV_PATH`: a CSV file, either a URL or a path in your Drive (for example `f"{DRIVE_DIR}/my_data.csv"`; upload the file to the `fine-tuning-workshop` folder first).
- `INPUT_COL`: the column the model reads.
- `TARGET_COL`: the column the model should learn to write.
- `SYSTEM_PROMPT`: an instruction given with every input, or `None`. For labels, list them here, so the model knows its options before training.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/UoA-eResearch/fine-tuning_workshop/main/data"

CSV_PATH = f"{DATA_URL}/arxiv_abstracts.csv"
INPUT_COL = "abstract"
TARGET_COL = "field"
SYSTEM_PROMPT = ("Which field is this research abstract from? Reply with one of: "
                 "Astrophysics, Quantitative biology, Economics, Mathematics.")

### 2b. Check the data

Most fine-tuning problems are data problems. This cell checks for:
- **empty rows** and **duplicate inputs** (removed),
- **inputs too long** for the model (counted in **tokens**, the word pieces the model reads),
- **class balance**, if the target is a label: if 90% of rows have one label, a model that always says that label scores 90%.

Problems are printed as `WARNING` lines. The template treats the target as a **label** if it has 20 or fewer distinct values, and as **free text** otherwise.

In [ ]:
import pandas as pd
from transformers import AutoTokenizer

MAX_SEQ_LENGTH = 1024   # longest example (in tokens) used in training

df = pd.read_csv(CSV_PATH)
print(f"{len(df)} rows. Columns: {list(df.columns)}")

# Empty rows and duplicates
df = df[[INPUT_COL, TARGET_COL]]
empty = df.isna().any(axis=1) | (df.astype(str).apply(lambda col: col.str.strip()) == "").any(axis=1)
if empty.sum():
    print(f"WARNING: removed {empty.sum()} rows with an empty input or target")
df = df[~empty]
dupes = df.duplicated(subset=INPUT_COL)
if dupes.sum():
    print(f"WARNING: removed {dupes.sum()} rows with a duplicate input")
df = df[~dupes].astype(str).reset_index(drop=True)

# Length in tokens
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
lengths = [len(ids) for ids in tokenizer((df[INPUT_COL] + " " + df[TARGET_COL]).tolist())["input_ids"]]
too_long = sum(n > MAX_SEQ_LENGTH - 100 for n in lengths)   # leave room for the prompt and markers
print(f"Tokens per example: median {pd.Series(lengths).median():.0f}, max {max(lengths)}")
if too_long:
    print(f"WARNING: {too_long} examples are too long and will be cut off in training")

# Labels or free text?
IS_LABEL = df[TARGET_COL].nunique() <= 20
if IS_LABEL:
    LABELS = sorted(df[TARGET_COL].unique())
    counts = df[TARGET_COL].value_counts()
    print(f"\nTarget looks like a label ({len(LABELS)} classes):")
    print(counts.to_string())
    if counts.min() < 0.5 * counts.max():
        print("WARNING: classes are unbalanced; compare your accuracy with always guessing the biggest class")
    if counts.min() < 20:
        print("WARNING: some classes have fewer than 20 examples")
else:
    print(f"\nTarget looks like free text ({df[TARGET_COL].nunique()} distinct values)")

print(f"\n{len(df)} rows after cleaning")

### 2c. Convert to chat format

Each row becomes the same `{"messages": [...]}` format as the pirate dataset: the input is the user's turn, and the target is the assistant's reply. If you set a system prompt, it goes first.

In [ ]:
import json

def to_messages(input_text, target_text=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] if SYSTEM_PROMPT else []
    messages.append({"role": "user", "content": input_text})
    if target_text is not None:
        messages.append({"role": "assistant", "content": target_text})
    return messages

df["messages"] = [to_messages(i, t) for i, t in zip(df[INPUT_COL], df[TARGET_COL])]
print(json.dumps({"messages": df["messages"][0]}, indent=2))

### 2d. Train, validation and test sets

The same three sets as Notebook 1 (section 2e): 70% train, 10% validation, 20% test. For labels, the split is **stratified**: each set gets the same mix of labels.

The sets are saved to `fine-tuning-workshop/your_data` in Drive. The fixed seed gives the same split every time you run this. If you add more data later, keep the old test file, so your scores stay comparable.

In [ ]:
from sklearn.model_selection import train_test_split

stratify = df[TARGET_COL] if IS_LABEL and df[TARGET_COL].value_counts().min() >= 5 else None
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=stratify)
stratify = train_df[TARGET_COL] if stratify is not None else None
train_df, val_df = train_test_split(train_df, test_size=0.125, random_state=SEED, stratify=stratify)   # 10% of all rows

DATA_DIR = f"{DRIVE_DIR}/your_data"
os.makedirs(DATA_DIR, exist_ok=True)
for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part[["messages"]].to_json(f"{DATA_DIR}/{name}.jsonl", orient="records", lines=True)

print(f"Training: {len(train_df)}   Validation: {len(val_df)}   Test: {len(test_df)}")
print(f"Saved to {DATA_DIR}")

*In the workshop, stop here. The rest of section 2 trains the model: run it at home on your own data.*

---

### 2e. Baseline: the model before training

Load the base model and run it on the test set, with your system prompt. This is the **prompting baseline** from Notebook 1: if prompting alone is good enough, you don't need to fine-tune.

For labels, each reply is matched to a label: a reply that names exactly one label counts as that label, and anything else counts as `(none)`.

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False)
FastLanguageModel.for_inference(model)

test_inputs = test_df[INPUT_COL].tolist()
test_targets = test_df[TARGET_COL].tolist()
MAX_NEW_TOKENS = 20 if IS_LABEL else 300   # labels are short; raise for long free-text targets

def to_label(reply):
    found = [label for label in LABELS if label.lower() in reply.lower()]
    return found[0] if len(found) == 1 else "(none)"

def accuracy(replies):
    return sum(to_label(r) == t for r, t in zip(replies, test_targets)) / len(test_targets)

before = ask_many(model, tokenizer, test_inputs, system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)
if IS_LABEL:
    print(f"Accuracy before training: {accuracy(before):.0%}")
    print(f"Always guessing the biggest class: {pd.Series(test_targets).value_counts().iloc[0] / len(test_targets):.0%}")
for i, t, r in list(zip(test_inputs, test_targets, before))[:3]:
    print(f"\nINPUT: {i[:200]}...\nTARGET: {t}\nREPLY: {r}")

### 2f. Train

The same LoRA set-up and trainer as Notebook 1 (sections 3 and 5). Three settings are worth changing:
- `MAX_STEPS`: how many training steps. Raise it if the validation loss is still falling at the end; lower it if the validation loss starts rising (overfitting).
- `LEARNING_RATE`: how big each update is. Lower it (e.g. `1e-4`) if the loss jumps around or the model starts producing nonsense.
- `LORA_RANK`: the size of the adapters. Try 8 for small or simple tasks, 32 if the model isn't learning enough.

With 280 training rows and 8 rows per step, 60 steps is about 1.7 passes through the data.

In [ ]:
MAX_STEPS = 60          # try 30 or 120
LEARNING_RATE = 2e-4    # try 1e-4 or 5e-5
LORA_RANK = 16          # try 8 or 32

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth import FastModel
from unsloth.chat_templates import train_on_responses_only

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

def to_dataset(part):
    texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in part["messages"]]
    return Dataset.from_dict({"text": texts})

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=to_dataset(train_df),
    eval_dataset=to_dataset(val_df),
    args=SFTConfig(
        per_device_train_batch_size=8,
        max_steps=MAX_STEPS,
        learning_rate=LEARNING_RATE,
        max_length=MAX_SEQ_LENGTH,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=20,
        seed=SEED,
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

stats = trainer.train()
print(f"Training took {stats.metrics['train_runtime']:.0f} seconds")

> **DRAFT NOTE:** Inherits Notebook 1's open T4 checks: `train_on_responses_only` markers for Qwen3.5 (with a system turn here), fp16 chosen automatically, batch size 8 fitting in memory, and `max_length` as the `SFTConfig` name for the sequence length in TRL ≤1.15. Time 2e–2h on the sample data.

The training loss should fall. If the validation loss starts rising while the training loss keeps falling, the model is overfitting.

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train = [(l["step"], l["loss"]) for l in logs if "loss" in l]
val = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.plot(*zip(*train), label="training loss")
if val:
    plt.plot(*zip(*val), "o-", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

### 2g. Evaluate: before vs after

The same test set, the same system prompt. For labels you get accuracy and a **confusion matrix**: a table of true labels (rows) against predicted labels (columns), so you can see which classes get mixed up. For free text you get the answers side by side, to read and judge.

In [ ]:
FastLanguageModel.for_inference(model)
after = ask_many(model, tokenizer, test_inputs, system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)

pd.set_option("display.max_colwidth", 300)
if IS_LABEL:
    print(f"Accuracy before: {accuracy(before):.0%}   after: {accuracy(after):.0%}")
    display(pd.crosstab(pd.Series(test_targets, name="true"),
                        pd.Series([to_label(r) for r in after], name="predicted")))
else:
    display(pd.DataFrame({"input": test_inputs, "target": test_targets,
                          "before": before, "after": after}).head(10))

**Think about it:** which classes does the model confuse, and would a human expert confuse them too? Papers that sit between two fields (for example, mathematical biology) are hard for everyone.

### 2h. Apply the model to new rows

This is how you'd use the model in research: run it over new data and get a CSV back. `NEW_CSV_PATH` needs the same `INPUT_COL` as your training data, and no target column. The results are saved to Drive, ready to open in Excel, R or pandas.

In [ ]:
NEW_CSV_PATH = f"{DATA_URL}/arxiv_abstracts_new.csv"

new_df = pd.read_csv(NEW_CSV_PATH)
replies = ask_many(model, tokenizer, new_df[INPUT_COL].astype(str).tolist(),
                   system_prompt=SYSTEM_PROMPT, max_new_tokens=MAX_NEW_TOKENS)
new_df[f"predicted_{TARGET_COL}"] = [to_label(r) for r in replies] if IS_LABEL else replies

new_df.to_csv(f"{DATA_DIR}/predictions.csv", index=False)
print(f"Saved {len(new_df)} predictions to {DATA_DIR}/predictions.csv")
new_df.head(10)

### 2i. Save the adapter

Saved to Drive, next to your data. To use it later, load it as in section 1a.

In [ ]:
model.save_pretrained(f"{DATA_DIR}/lora")
tokenizer.save_pretrained(f"{DATA_DIR}/lora")
!ls -lh {DATA_DIR}/lora

## 3. Pitfalls checklist

Check these before you trust (or publish) a fine-tuned model.

| Pitfall | How to spot it | What to do |
|---|---|---|
| **Data leakage**: test examples, or near-copies of them, are in the training data | Test scores look too good to be true | Remove duplicates before splitting (2b). Split by group when rows are related: same patient, same paper, same chromosome (as in Notebook 2's test set) |
| **Overfitting**: the model memorises the training data | Training loss falls while validation loss rises | Fewer steps, more data, or a smaller rank |
| **Catastrophic forgetting**: the model loses things it could do before | General questions now get wrong answers (Notebook 1, section 6c) | Fewer steps, a lower learning rate, or mix some general examples into the training data |
| **No baseline** | You can't say whether fine-tuning helped | Always measure the untrained model, a prompted model, and a simple guess such as the most common label |
| **Results you can't reproduce** | A rerun gives different numbers, or you can't remember the settings | Record the model name and version, seed, settings, package versions and data version. Keep the test file fixed |

## 4. Where to go next

### More computing power

- **University of Auckland:** the [advanced compute page](https://www.auckland.ac.nz/en/research/research-resources/research-ai-software-computing/advanced-compute.html) lists the options. The most relevant is the **[Local AI Service (LAIS)](https://www.auckland.ac.nz/en/research/research-resources/research-ai-software-computing/advanced-compute/local-ai-service.html)**:
  - Dedicated Dell Pro Max GB10 machines with 128 GB of memory (models up to ~200 billion parameters), or 256 GB with two units (~400 billion).
  - Free. Your data and models stay on University premises, and machines are wiped between groups.
  - SSH access with CUDA, JupyterLab and Docker. Book from one week, extendable to four.
  - Staff and doctoral candidates request access directly; other students go through their supervisor. Contact researchdata@auckland.ac.nz.
  - This workshop is adapted from [ahoy_training](https://github.com/UoA-eResearch/ahoy_training), which was built for the same GB10 hardware, so its full-size demos are a natural next step.
- **Sensitive data** (for example, clinical notes): use the **Secure Research Environment (SRE)**, on the same advanced compute page.
- **Off campus:** free Colab has daily GPU limits; Colab Pro and Kaggle notebooks give more GPU time.

> **DRAFT NOTE:** The LAIS page describes "local LLM inference and testing" and doesn't mention fine-tuning. Confirm with the Centre for eResearch that fine-tuning is supported before presenting it this way.

### What changes with bigger models

- **Memory:** roughly 2 bytes per parameter in 16-bit, about 0.5 bytes in 4-bit. An 8B model needs ~16 GB in 16-bit, more than a T4 has, so **QLoRA** (LoRA on a 4-bit model) becomes necessary.
- **Speed:** training and generation are slower, and very large models may need several GPUs. ahoy_training fine-tunes a 72B model with QLoRA, as a concrete example.
- **Results:** bigger models usually do better, and often need fewer examples.

### How other researchers have used fine-tuning

**Discussed in the session**

1. **Chemistry: molecular properties** ([Jablonka et al., 2024, *Nature Machine Intelligence*](https://doi.org/10.1038/s42256-023-00788-1)). GPT-3 fine-tuned to answer chemistry and materials questions in plain language. It matched or beat conventional machine learning, especially with little data, and could suggest new molecules by asking the questions in reverse. *Link:* research datasets are usually small.
2. **Medicine: extracting data from clinical reports** ([*Scientific Reports*, 2025](https://doi.org/10.1038/s41598-025-28767-z)). Llama-3.1-8B fine-tuned with LoRA did as well as a second human annotator on four datasets, using desktop hardware and 100 or fewer training reports. *Link:* it runs locally, which suits sensitive data and services like LAIS.
3. **Radiology: a teacher model labels the data** ([Chen et al., 2024, BURExtract-Llama](https://arxiv.org/abs/2408.11334); workshop paper). GPT-4 labelled a small dataset, then Llama was fine-tuned on it and matched GPT-4 at lower cost and with better privacy. *Link:* the same teacher → student recipe as the pirate dataset.
4. **Linguistics and data sovereignty: te reo Māori speech recognition** (Te Hiku Media / Papa Reo). Whisper fine-tuned on data under their Kaitiakitanga Licence performed much better, but they chose not to use it in production because of ethical concerns and the potential harm to Indigenous languages. *Link:* being able to fine-tune a model doesn't mean you should; who owns the data matters.

**More to read**

5. **Materials science: from papers to databases** ([Dagdelen et al., 2024, *Nature Communications*](https://doi.org/10.1038/s41467-024-45563-x)). GPT-3 and Llama-2 fine-tuned to extract structured records (sentences or JSON) from papers, with around 500 examples. *Link:* a few hundred examples can be enough.
6. **Political science: annotating text** ([Alizadeh et al., 2024, *Journal of Computational Social Science*](https://arxiv.org/abs/2307.02179)). Open-source models fine-tuned for stance, topic and relevance in news and tweets matched or beat GPT-3.5 and GPT-4 without fine-tuning, and beat few-shot prompting. The authors share a notebook. *Link:* prompting vs fine-tuning, as in Notebook 1.
7. **Systematic reviews: screening titles and abstracts** ([Yamoah et al., 2026](https://arxiv.org/abs/2603.24767); preprint). A 1.2B model fine-tuned on more than 8,500 human-rated titles and abstracts improved a lot and missed few relevant papers, so it could act as a second screener. *Link:* a model about the size of the pirate model.
8. **Humanities: correcting OCR errors in historical newspapers** (Thomas et al., 2024, LT4HALA workshop). Instruction-tuned Llama 2 compared with fine-tuned BART for finding and fixing OCR errors in 19th-century British newspapers (BLN600). *Link:* a style and format task, like the pirate example.
9. **Education: essay scoring.** Human-AI Collaborative Essay Scoring fine-tuned Llama-3-8B (4-bit) with Unsloth and LoRA at rank 16 and alpha 16, almost the settings used here. [AiAWE (2026)](https://arxiv.org/abs/2606.12801) scores argumentative essays with a LoRA-adapted Gemma-3-27B. *Link:* the same tools as Notebook 1.
10. **Astronomy: adapting to a whole field** ([Nguyen et al., 2023, AstroLLaMA](https://arxiv.org/abs/2309.06126); preprint). LLaMA-2-7B trained further on more than 300,000 astronomy abstracts. *Link:* the contrast case: teaching a model a whole field, rather than one task, needs far more data and computing.

> **DRAFT NOTE:** Check every example against the original paper before publishing (claims and numbers came from summaries). Still needed: authors for the 2025 *Scientific Reports* clinical paper; a link for Thomas et al. (LT4HALA 2024); the full citation for Human-AI Collaborative Essay Scoring; and the **original Papa Reo blog post** for example 4 (not the summary it came from).

### Other ways to train

- **DPO** (Direct Preference Optimisation): instead of example answers, you give pairs of answers marked better and worse, and the model learns to prefer the better kind.
- **GRPO** (Group Relative Policy Optimisation): the model generates several answers, a scoring function you write rates them, and the model learns from the best. Used to train reasoning on tasks with checkable answers, such as maths.

Unsloth supports both.

### Links

- [Unsloth documentation](https://docs.unsloth.ai) and [Unsloth notebooks](https://github.com/unslothai/notebooks) (many models and tasks, ready for Colab)
- [Hugging Face LLM course](https://huggingface.co/learn/llm-course)
- [This workshop's materials](https://github.com/UoA-eResearch/fine-tuning_workshop)

## Recap

- **Keep your work:** save adapters to Drive; export to GGUF to run the model on your own computer with Ollama.
- **Your own data:** a CSV with an input and a target column is enough to start. Check it, split it, measure a baseline, then train.
- **Watch for** leakage, overfitting, forgetting and missing baselines, and record what you did.
- **Next steps:** the University's Local AI Service for bigger models, and the research examples above for ideas.